# Notebook 25 — Prompting, Context Engineering, and Chat Templates

    ## Learning objectives

    - Design prompts as versioned probabilistic interfaces
- Render and test Hugging Face chat templates
- Evaluate few-shot examples, context budgets, and injection boundaries

    Cells labeled **optional GPU/remote** are deliberately guarded. Read them first,
    then opt in when the required hardware or Hugging Face Inference access is available.


In [ ]:
# Colab/local environment setup — run this cell first.
import importlib.util
import os
import platform
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
PACKAGES = ['transformers>=4.51,<5', 'sentencepiece']

if IN_COLAB and PACKAGES:
    print("Installing notebook dependencies in the Colab runtime...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *PACKAGES])

# Load an HF token from Colab Secrets without displaying it. In Colab, create a
# secret named HF_TOKEN (or HUGGINGFACE_TOKEN) and enable notebook access.
if IN_COLAB:
    from google.colab import userdata
    token = None
    for secret_name in ("HF_TOKEN", "HUGGINGFACE_TOKEN"):
        try:
            token = userdata.get(secret_name)
        except Exception:
            pass
        if token:
            break
    if token:
        os.environ["HF_TOKEN"] = token
        os.environ["HUGGINGFACE_TOKEN"] = token
else:
    try:
        from dotenv import load_dotenv
        load_dotenv(".env")
    except ImportError:
        pass

try:
    import torch
    accelerator = torch.cuda.get_device_name(0) if torch.cuda.is_available() else (
        "Apple MPS" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "CPU"
    )
    print(f"runtime={platform.platform()} | Python={platform.python_version()} | accelerator={accelerator}")
    if False and not torch.cuda.is_available():
        print("WARNING: this training notebook is designed for a Colab GPU runtime. "
              "Select Runtime > Change runtime type > T4 GPU (or better).")
except ImportError:
    print(f"runtime={platform.platform()} | Python={platform.python_version()}")

print("Hugging Face token configured:", bool(os.getenv("HUGGINGFACE_TOKEN")))


## 25.1 Prompting is interface design

A prompt conditions a trained distribution; it does not install a rule or permission boundary. State the task, audience, supplied data, constraints, uncertainty behavior, and output contract. Use delimiters to distinguish instructions from untrusted data, but assume the model may still follow injected text. Positive instructions often work better than lists of prohibitions. Establish a zero-shot baseline before decomposition or few-shot examples, and measure success on frozen cases rather than optimizing one anecdote.


In [ ]:
cases=[{"input":"refund policy","expected_fields":{"answer","sources"}},{"input":"unknown policy","must_abstain":True}]
print(cases)


## 25.2 Few-shot and context engineering

Examples teach local patterns without changing weights, but selection, order, label balance, similarity, and length affect behavior. Keep examples free of evaluation leakage and retrieve them by task-relevant similarity only under authorization. Context engineering chooses all messages, evidence, tool schemas, memory, and prior results under a token budget. Preserve authoritative constraints and unresolved commitments; summarize low-priority history with provenance. More context can distract, increase latency, and expose injection.


In [ ]:
examples=[("positive","Return one JSON object with answer and sources."),("negative","Do not ramble or omit JSON or add markdown.")]
print(*examples,sep="\n")


## 25.3 Chat templates are model artifacts

Instruct checkpoints are trained on exact control tokens and role serialization. `apply_chat_template(tokenize=True)` is safest because rendering then ordinary tokenization can duplicate special tokens. Understand `add_generation_prompt` and continuing a final message, EOS boundaries, assistant masks, tool arguments, and reasoning controls. Templates are Jinja stored with tokenizer artifacts; whitespace changes can matter. Multimodal templates belong to processors and emit media placeholders. Pin, inspect, and regression-test rendered IDs across training and every serving engine.


In [ ]:
try:
 from transformers import AutoTokenizer
 tok=AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct",local_files_only=True); messages=[{"role":"user","content":"Define perplexity."}]; ids=tok.apply_chat_template(messages,tokenize=True,add_generation_prompt=True); print(tok.decode(ids),len(ids))
except Exception as e: print("Load/cache the public tokenizer to inspect its exact template:",type(e).__name__)


## 25.4 Prompt evaluation and escalation

Version prompt source, variables, rendered text hash, template/model/tokenizer revisions, and decoding. Evaluate correctness, constraint adherence, schema validity, abstention, robustness to irrelevant changes, injection, token cost, and latency. Use paired comparisons and slice results. When prompt complexity grows, decide whether constrained decoding, retrieval, tools, SFT, or deterministic code owns the requirement. Prompts cannot guarantee authorization, factuality, confidentiality, or exact computation; application controls remain authoritative.


In [ ]:
results=[{"prompt":"v1","valid":8,"correct":6,"tokens":900},{"prompt":"v2","valid":10,"correct":7,"tokens":1300}]
for r in results: print(r["prompt"],"accuracy",r["correct"]/10,"tokens/correct",r["tokens"]/r["correct"])


## Reference workflow and evidence standard

Treat the notebook as an experiment, not a recipe. State the question, freeze inputs and
success criteria, establish the simplest baseline, change one material factor, and retain raw
outputs needed to diagnose failures. Record model, tokenizer, template, data and code revisions;
hardware and dtype; random seeds; generation or optimization configuration; token counts;
latency and memory; and results by meaningful slice. A demonstration that runs is evidence of
plumbing, not evidence of general capability.

Test boundaries as well as the happy path: empty and maximum-length inputs, malformed records,
multilingual or code text, unavailable dependencies, cancellation, and adversarial content.
Keep credentials in environment or Colab Secrets and never serialize them with artifacts. Pin
remote revisions, review licenses and custom code, validate saved artifacts in a fresh process,
and prefer deterministic validators wherever outputs can be checked mechanically.

Before applying the technique, compare it with prompting, retrieval, a smaller model, or no
model. Report quality together with compute, storage, latency, and operational complexity. Use
held-out data and paired comparisons, disclose uncertainty and negative results, and define a
rollback path. These practices connect low-level understanding to reliable application work.

A useful completion checklist asks four separate questions. Is the mathematical contract clear
enough to predict shapes, masks, reductions, and failure cases? Does the implementation reproduce
a tiny hand-worked or deterministic reference? Does the measured result survive a held-out set,
relevant slices, and an ablation against a simpler baseline? Can another person reload the exact
artifacts and reconstruct the claim from the manifest? Passing only the first two establishes a
tutorial demonstration; passing all four supports an engineering decision. When a result fails,
preserve the counterexample and update the test suite before changing the implementation.

Finally, separate correctness, capability, efficiency, and safety conclusions. A correct
implementation may have weak capability; a capable prototype may be too costly or unsafe to
deploy. Name the population to which each conclusion applies and avoid converting a single
metric into a universal ranking. Track assumptions beside results, especially tokenizer and
template compatibility, data rights, access-control boundaries, and hardware-specific behavior.
Leave exercises with an executable acceptance criterion, a baseline result, and a short written
interpretation. That combination turns exploratory code into cumulative course evidence that can
be revisited when libraries, model families, or deployment engines change.


## 25.5 Rendered-prompt regression tests

Treat the rendered prompt as an artifact. Freeze representative conversations containing system messages, multiple turns, tool calls, empty content, Unicode, and adversarial delimiters; assert exact token IDs or stable hashes for a pinned tokenizer revision. Test `add_generation_prompt` separately from continuing an assistant message. Rendering text and tokenizing again can duplicate BOS or EOS, so prefer template tokenization directly. A template change can alter both evaluation and fine-tuning behavior even when user-visible messages are identical.


In [ ]:
import hashlib
def token_hash(ids): return hashlib.sha256(bytes(str(list(ids)),"utf-8")).hexdigest()[:16]
fixtures={"single_turn":[1,10,20,2,30],"multi_turn":[1,10,20,2,30,40,2,30]}; print({k:token_hash(v) for k,v in fixtures.items()})


## 25.6 Prompt ablations and robustness

Compare prompts on a frozen set using paired examples, not aggregate anecdotes. Ablate system wording, delimiters, few-shot examples, order, irrelevant context, and output instructions one factor at a time. Measure correctness, schema validity, abstention, token count, latency, and injection resistance by slice. A longer prompt that gains one metric may be worse after cost and fragility. Move requirements that must always hold—authorization, validation, calculations, and access control—into deterministic application code.


In [ ]:
runs=[{"case":"a","v1":1,"v2":1},{"case":"b","v1":0,"v2":1},{"case":"c","v1":1,"v2":0},{"case":"d","v1":0,"v2":1}]
deltas=[r["v2"]-r["v1"] for r in runs]; print("paired delta",sum(deltas)/len(deltas),"wins/losses",deltas.count(1),deltas.count(-1))


## 25.7 Template files, variants, and portability

Modern Transformers repositories save the default template as a reviewable `chat_template.jinja`; named alternatives, such as a tool-use template, can live under `additional_chat_templates/<name>.jinja`. Older repositories may embed template text in tokenizer or processor configuration. Treat all forms as versioned model artifacts. Save and reload the tokenizer or processor in a clean directory, enumerate available variants, and regression-test rendered tokens for each supported path. Do not assume a serving engine reads the same file or selects the same named template: make template ownership explicit and run parity fixtures across local Transformers, hosted inference, Ollama, vLLM, and any gateway.


In [ ]:
from pathlib import PurePosixPath
template_artifacts=[PurePosixPath("chat_template.jinja"),PurePosixPath("additional_chat_templates/tool_use.jinja")]
for artifact in template_artifacts: print(artifact,"variant",artifact.stem)
parity_contract={"default":"token-hash-a","tool_use":"token-hash-b"}; print(parity_contract)


## Primary references and further study

Use the pinned library documentation that matches your environment. Papers explain the method and assumptions; current official documentation defines the executable API.

- [Transformers chat templates](https://huggingface.co/docs/transformers/chat_templating)
- [Writing chat templates](https://huggingface.co/docs/transformers/chat_templating_writing)


## Exercises

    1. Compare zero-shot and few-shot prompts on a frozen set.
2. Write and test a minimal Jinja chat template.
3. Build a token-budgeted context assembler.

    ## Checkpoint

    Explain the notebook's central mechanism without using library names, then identify
    one assumption you would test before applying it to a real workload.
